# Event Hub Streaming Consumer

This notebook reads real-time TRISTAR public transport vehicle events from Gdańsk through Azure Event Hubs and writes them to a Bronze Delta table.

## Load configuration

Load settings from the YAML configuration file.

In [0]:
import yaml

with open("config.yml", "r") as file:
    config = yaml.safe_load(file)

## Get parameters

Read Job parameters and Event Hub configuration.

In [0]:
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")
event_hub_name = dbutils.widgets.get("event_hub_name")

event_hub_namespace = config["event_hub"]["namespace"]
scope = config["event_hub"]["secret_scope"]

## Define paths and secrets


In [0]:
bronze_schema = f"{schema}_bronze"
bronze_table = f"{catalog}.{bronze_schema}.tristar_events"

checkpoint_path = (
    f"/Volumes/{catalog}/{schema}/raw_files/"
    "tristar_eventhub/checkpoint"
)
bronze_checkpoint_path = f"{checkpoint_path}/bronze"

event_hub_connection_str = dbutils.secrets.get(
    scope=scope,
    key="evh-connection-string"
)

## Configure Event Hub connection

Create the Kafka connection settings for Azure Event Hubs.

In [0]:
bootstrap_servers = f"{event_hub_namespace}.servicebus.windows.net:9093"

sasl_config = (
    'kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule '
    f'required username="$ConnectionString" '
    f'password="{event_hub_connection_str}";'
)

## Read streaming data

In [0]:
raw_df = (
    spark.readStream
        .format("kafka")
        .option("kafka.bootstrap.servers", bootstrap_servers)
        .option("subscribe", event_hub_name)
        .option("kafka.security.protocol", "SASL_SSL")
        .option("kafka.sasl.mechanism", "PLAIN")
        .option("kafka.sasl.jaas.config", sasl_config)
        .option("startingOffsets", "latest")
        .load()
)

## Prepare Bronze data

Select the raw payload and Event Hub metadata and add the ingestion timestamp.

In [0]:
from pyspark.sql.functions import col, current_timestamp, lit

bronze_df = (
    raw_df
    .selectExpr(
        "CAST(value AS STRING) AS raw_payload",
        "partition",
        "offset",
        "timestamp AS eh_enqueued_timestamp"
    )
    .withColumn("ingestion_timestamp", current_timestamp())
)

## Write to Bronze

Write the streaming data to the Bronze Delta table using checkpointing.

In [0]:
query = (
    bronze_df.writeStream
    .format("delta")
    .option("checkpointLocation", bronze_checkpoint_path)
    .trigger(availableNow=True)
    .toTable(bronze_table)
)

query.awaitTermination()